# 🧠 Milestone 3 — CNN from Scratch (PyTorch)
**Train a Convolutional Neural Network on Mel-Spectrograms**

### What you'll learn:
- PyTorch basics: Tensors, Dataset, DataLoader
- How to convert audio → 2D Mel-Spectrogram images
- How to build a CNN architecture from scratch
- Training loop with loss, optimizer, validation
- Full W&B experiment tracking

### Why CNN works for audio?
Mel-Spectrograms are 2D images (time × frequency). CNNs are great at finding local patterns in images, so they naturally capture rhythmic and timbral patterns in audio!

In [ ]:
!pip install librosa wandb -q
# PyTorch is pre-installed on Kaggle

In [ ]:
import os, random, warnings, time
import numpy as np
import pandas as pd
import librosa
import wandb
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Using device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')

BASE        = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'
STEMS_DIR   = f'{BASE}/genres_stems'
MASHUPS_DIR = f'{BASE}/mashups'
TEST_CSV    = f'{BASE}/test.csv'
GENRES      = ['blues','classical','country','disco','hiphop','jazz','metal','pop','reggae','rock']
ROLL_NO     = 'YOUR_ROLL_NO'  # ⚠️ change this!

## 1️⃣ Audio → Mel-Spectrogram
We convert each audio file into a **fixed-size image** (128 × 256 pixels).
Each pixel = energy at a certain frequency at a certain time.

In [ ]:
# ── Hyperparameters (tune these!) ─────────────────────────────────────────────
CONFIG = {
    'sample_rate'  : 22050,
    'duration'     : 30,       # seconds to load per file
    'n_mels'       : 128,      # mel frequency bins (height of image)
    'n_fft'        : 2048,     # FFT window size
    'hop_length'   : 512,      # step between FFT windows
    'target_length': 256,      # fixed time frames (width of image)
    'batch_size'   : 32,
    'epochs'       : 30,
    'lr'           : 1e-3,
    'dropout'      : 0.3,
    'model'        : 'CNN_from_scratch',
}

def audio_to_melspec(path, cfg=CONFIG):
    """
    Load audio and return a normalized mel-spectrogram.
    Output shape: (1, n_mels, target_length) — single channel image.
    """
    try:
        y, sr = librosa.load(path, sr=cfg['sample_rate'], duration=cfg['duration'])
        if len(y) < sr:
            y = np.zeros(sr * cfg['duration'])
        mel    = librosa.feature.melspectrogram(
                    y=y, sr=sr, n_mels=cfg['n_mels'],
                    n_fft=cfg['n_fft'], hop_length=cfg['hop_length'])
        mel_db = librosa.power_to_db(mel, ref=np.max)  # convert to dB scale
        # Pad or crop to fixed width
        T = cfg['target_length']
        if mel_db.shape[1] < T:
            mel_db = np.pad(mel_db, ((0,0),(0, T - mel_db.shape[1])))
        else:
            mel_db = mel_db[:, :T]
        # Normalize to [0, 1]
        mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-8)
        return mel_db[np.newaxis, :, :]  # add channel dim → (1, 128, 256)
    except Exception as e:
        return np.zeros((1, CONFIG['n_mels'], CONFIG['target_length']))

# Quick test
sample_path = f"{STEMS_DIR}/blues/{sorted(os.listdir(f'{STEMS_DIR}/blues'))[0]}/vocals.wav"
spec = audio_to_melspec(sample_path)
print(f'Mel-spec shape: {spec.shape}  (channels, freq_bins, time_frames)')

plt.figure(figsize=(10,3))
plt.imshow(spec[0], aspect='auto', origin='lower', cmap='inferno')
plt.colorbar(); plt.title('Mel-Spectrogram (blues/vocals)')
plt.xlabel('Time'); plt.ylabel('Mel Frequency')
plt.tight_layout(); plt.show()

## 2️⃣ Build Dataset
PyTorch `Dataset` is just a class with `__len__` and `__getitem__` — it tells the DataLoader how to fetch one sample.

In [ ]:
class AudioDataset(Dataset):
    """Dataset for training stems. Randomly picks ONE stem per song per call."""
    def __init__(self, file_list, label_list, augment=False):
        self.files   = file_list
        self.labels  = label_list
        self.augment = augment

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        path  = self.files[idx]
        label = self.labels[idx]
        mel   = audio_to_melspec(path)

        if self.augment:
            # Time masking — randomly zero out time frames
            t_start = random.randint(0, CONFIG['target_length'] - 30)
            mel[0, :, t_start:t_start+20] = 0
            # Frequency masking — randomly zero out freq bands
            f_start = random.randint(0, CONFIG['n_mels'] - 15)
            mel[0, f_start:f_start+10, :] = 0

        return torch.FloatTensor(mel), torch.tensor(label, dtype=torch.long)

class TestDataset(Dataset):
    """Dataset for test mashup files."""
    def __init__(self, file_list):
        self.files = file_list
    def __len__(self):
        return len(self.files)
    def __getitem__(self, idx):
        mel = audio_to_melspec(self.files[idx])
        return torch.FloatTensor(mel)

print('✅ Dataset classes defined')

In [ ]:
# ── Build file lists and labels ───────────────────────────────────────────────
# Strategy: use vocals + drums stems for training (most genre-informative)
USE_STEMS = ['vocals', 'drums']   # you can add 'bass','others' for more data

all_files, all_labels = [], []
for genre in GENRES:
    songs = sorted(os.listdir(f'{STEMS_DIR}/{genre}'))
    for song in songs:
        for stem in USE_STEMS:
            path = f'{STEMS_DIR}/{genre}/{song}/{stem}.wav'
            if os.path.exists(path):
                all_files.append(path)
                all_labels.append(genre)

le = LabelEncoder()
all_labels_enc = le.fit_transform(all_labels)
print(f'Total training samples: {len(all_files)}')
print(f'Classes: {le.classes_}')

In [ ]:
# Train/val split (80/20)
X_tr, X_val, y_tr, y_val = train_test_split(
    all_files, all_labels_enc, test_size=0.2, stratify=all_labels_enc, random_state=42)

train_ds = AudioDataset(X_tr, y_tr, augment=True)
val_ds   = AudioDataset(X_val, y_val, augment=False)

train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'],
                          shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=CONFIG['batch_size'],
                          shuffle=False, num_workers=2, pin_memory=True)

print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')

## 3️⃣ CNN Architecture
Our CNN has 4 convolutional blocks followed by a classifier head.
Each block: Conv2D → BatchNorm → ReLU → MaxPool → Dropout

In [ ]:
class ConvBlock(nn.Module):
    """One convolutional block: Conv → BN → ReLU → Pool → Dropout."""
    def __init__(self, in_ch, out_ch, pool_size=(2,2), dropout=0.3):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(pool_size),
            nn.Dropout2d(dropout),
        )
    def forward(self, x):
        return self.block(x)


class GenreCNN(nn.Module):
    """
    CNN built from scratch for audio genre classification.
    Input : (batch, 1, 128, 256) mel-spectrogram
    Output: (batch, 10) class logits
    """
    def __init__(self, n_classes=10, dropout=0.3):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(1,  32, pool_size=(2,2), dropout=dropout),   # → (32, 64, 128)
            ConvBlock(32, 64, pool_size=(2,2), dropout=dropout),   # → (64, 32, 64)
            ConvBlock(64, 128, pool_size=(2,2), dropout=dropout),  # → (128, 16, 32)
            ConvBlock(128, 256, pool_size=(2,2), dropout=dropout), # → (256, 8, 16)
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),  # → (256, 1, 1) — handles any input size
            nn.Flatten(),                  # → (256,)
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, n_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


model = GenreCNN(n_classes=len(GENRES), dropout=CONFIG['dropout']).to(DEVICE)
print(model)

# Count parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\n✅ Trainable parameters: {total_params:,}')

## 4️⃣ Training Loop

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, all_preds, all_labels = 0, [], []
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        out  = model(X)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y)
        all_preds.extend(out.argmax(1).cpu().numpy())
        all_labels.extend(y.cpu().numpy())
    avg_loss = total_loss / len(loader.dataset)
    f1       = f1_score(all_labels, all_preds, average='macro')
    return avg_loss, f1

@torch.no_grad()
def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, all_preds, all_labels = 0, [], []
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        out  = model(X)
        loss = criterion(out, y)
        total_loss += loss.item() * len(y)
        all_preds.extend(out.argmax(1).cpu().numpy())
        all_labels.extend(y.cpu().numpy())
    avg_loss = total_loss / len(loader.dataset)
    f1       = f1_score(all_labels, all_preds, average='macro')
    return avg_loss, f1, all_preds, all_labels

print('✅ Training functions defined')

In [ ]:
# ── Initialize training ───────────────────────────────────────────────────────
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG['epochs'])

wandb.init(project=f'{ROLL_NO}-t12026', name='milestone3-cnn-scratch',
           config=CONFIG)
wandb.watch(model, log='all', log_freq=50)

best_val_f1   = 0
train_history = {'loss': [], 'f1': []}
val_history   = {'loss': [], 'f1': []}

print(f'Training for {CONFIG["epochs"]} epochs on {DEVICE}...')
print('='*60)

In [ ]:
for epoch in range(1, CONFIG['epochs'] + 1):
    t0 = time.time()
    tr_loss, tr_f1 = train_epoch(model, train_loader, optimizer, criterion)
    va_loss, va_f1, va_preds, va_labels = eval_epoch(model, val_loader, criterion)
    scheduler.step()

    train_history['loss'].append(tr_loss)
    train_history['f1'].append(tr_f1)
    val_history['loss'].append(va_loss)
    val_history['f1'].append(va_f1)

    # Save best model
    if va_f1 > best_val_f1:
        best_val_f1 = va_f1
        torch.save(model.state_dict(), 'best_cnn.pt')
        print(f'  💾 New best saved!')

    # W&B logging
    wandb.log({'epoch': epoch, 'train_loss': tr_loss, 'train_f1': tr_f1,
               'val_loss': va_loss, 'val_f1': va_f1,
               'lr': optimizer.param_groups[0]['lr']})

    print(f'Epoch {epoch:2d}/{CONFIG["epochs"]} | '
          f'Train Loss: {tr_loss:.4f} F1: {tr_f1:.4f} | '
          f'Val Loss: {va_loss:.4f} F1: {va_f1:.4f} | '
          f'Time: {time.time()-t0:.1f}s')

print(f'\n✅ Best Val Macro F1: {best_val_f1:.4f}')

## 5️⃣ Results & Analysis

In [ ]:
# Learning curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(train_history['loss'], label='Train', color='steelblue')
axes[0].plot(val_history['loss'],   label='Val',   color='tomato')
axes[0].set_title('Loss'); axes[0].legend()
axes[1].plot(train_history['f1'], label='Train', color='steelblue')
axes[1].plot(val_history['f1'],   label='Val',   color='tomato')
axes[1].axhline(0.80, color='green', linestyle='--', label='Target (0.80)')
axes[1].set_title('Macro F1'); axes[1].legend()
plt.suptitle('CNN Training Curves', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('cnn_curves.png', dpi=150); plt.show()
wandb.log({'training_curves': wandb.Image('cnn_curves.png')})

In [ ]:
# Load best model and get classification report
model.load_state_dict(torch.load('best_cnn.pt'))
_, best_f1, best_preds, best_labels = eval_epoch(model, val_loader, criterion)
print(f'Best model Val Macro F1: {best_f1:.4f}')
print('\nClassification Report:')
print(classification_report(best_labels, best_preds, target_names=le.classes_))
wandb.log({'best_val_macro_f1': best_f1})
wandb.finish()

## 6️⃣ Inference & Kaggle Submission

In [ ]:
test_df   = pd.read_csv(TEST_CSV)
fname_col = test_df.columns[1]
test_files = [f'{MASHUPS_DIR}/{row[fname_col]}' for _, row in test_df.iterrows()]
test_ds    = TestDataset(test_files)
test_loader= DataLoader(test_ds, batch_size=CONFIG['batch_size'],
                        shuffle=False, num_workers=2)

model.eval()
all_test_preds = []
with torch.no_grad():
    for X in test_loader:
        X    = X.to(DEVICE)
        out  = model(X)
        preds= out.argmax(1).cpu().numpy()
        all_test_preds.extend(preds)

pred_genres = le.inverse_transform(all_test_preds)
submission  = pd.DataFrame({'id': test_df['id'], 'genre': pred_genres})
submission.to_csv('submission_m3_cnn.csv', index=False)
print('✅ submission_m3_cnn.csv saved')
print('Distribution:')
print(submission['genre'].value_counts())

### 📌 What to expect
- Val Macro F1 ~ **0.55–0.70** with this CNN
- To reach 0.80+, move to **Milestone 4 (CRNN)** and **Milestone 5 (Transformer)**
- Key levers: more epochs, data augmentation, all 4 stems, larger model